# 🔀 Notebook 03 — Hybrider, kontekst og fairness

**ALS gir Lea bedre anbefalinger. Men fungerer det for *alle*?**

Nå går vi fra modell til produkt. Vi kombinerer langtidsprofil med korttidskontekst
og sjekker om systemet faktisk er godt nok for flere enn mainstream-brukerne.

## Hvorfor étt verktøy aldri er nok

Du bruker ikke bare en hammer til å bygge et hus. Samme med anbefalingssystemer —
ingen enkeltmodell løser hele produktproblemet:

- **Popularitet** er robust, men upersonlig
- **Innholdsbasert** hjelper ved nye items og tynne signaler
- **ALS** gir sterk personalisering når det finnes data
- **Kontekst og reranking** hjelper når brukerens situasjon eller produktkrav endrer seg

Hybrider lar oss kombinere disse styrkene.

## Oppsett

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, diags
from implicit.als import AlternatingLeastSquares
from src.data import load_interactions, load_item_metadata, load_sessions, get_genre_matrix
from src.split import leave_one_out_split, build_sparse_matrix, session_train_test_split
from src.metrics import recall_at_k, coverage, novelty
from src.fairness import popularity_bias_analysis, genre_calibration_score, group_recall_comparison
from src.rerank import mmr_rerank

interactions = load_interactions()
items = load_item_metadata()
sessions = load_sessions()
train_df, test_df = leave_one_out_split(interactions)
n_users = interactions.user_id.max() + 1
n_items = interactions.item_id.max() + 1
train_matrix = build_sparse_matrix(train_df, n_users, n_items)
genre_matrix = get_genre_matrix(items)
user_ids = test_df['user_id'].values
test_items = test_df['item_id'].values
K = 10
LEA_ID = 451

als = AlternatingLeastSquares(factors=64, regularization=0.01, iterations=15, random_state=42, use_gpu=False)
als.fit(train_matrix, show_progress=True)
recs_als = als.recommend(user_ids, train_matrix[user_ids], N=K, filter_already_liked_items=True)[0]
item_pop_counts = np.asarray(train_matrix.sum(axis=0)).flatten()
item_pop_frac = item_pop_counts / n_users

print('Oppsett ferdig')

## 🏋️ Oppgave 4 — Korttidskontekst som hybridsignal

In [ ]:
sess_sizes = sessions.groupby('session_id').size()
print(f'Antall sesjoner: {len(sess_sizes):,}')
print(f'Sesjonslengde — min: {sess_sizes.min()}, median: {sess_sizes.median():.0f}, maks: {sess_sizes.max()}')

lea_sessions = sessions[sessions.user_id == LEA_ID]
lea_session_ids = lea_sessions.session_id.unique()
print(f'Lea har {len(lea_session_ids)} sesjoner')

In [ ]:
train_sessions, test_sessions = session_train_test_split(sessions, interactions)
counts = lil_matrix((n_items, n_items), dtype=np.float64)
sorted_sessions = train_sessions.sort_values(['session_id', 'position'])
prev_item, prev_session = -1, -1
for _, row in sorted_sessions.iterrows():
    session_id, item_id = row['session_id'], row['item_id']
    if session_id == prev_session and prev_item >= 0:
        counts[prev_item, item_id] += 1
    prev_item, prev_session = item_id, session_id
transition = counts.tocsr()
row_sums = np.asarray(transition.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0
transition = diags(1.0 / row_sums) @ transition

def recommend_session_blend(als_model, transition, test_sessions, train_matrix, k=10, lambda_=0.7):
    dense_transition = transition.toarray() if hasattr(transition, 'toarray') else transition
    recommendations, ground_truth = [], []
    for _, group in test_sessions.groupby('session_id'):
        group = group.sort_values('position')
        if len(group) < 2:
            continue
        user_id = group['user_id'].iloc[0]
        context_items = group['item_id'].values[:-1].tolist()
        target_item = group['item_id'].values[-1]

        als_scores = als_model.user_factors[user_id] @ als_model.item_factors.T
        als_scores = (als_scores - als_scores.min()) / max(als_scores.max() - als_scores.min(), 1e-10)

        session_scores = np.zeros(dense_transition.shape[0])
        weight = 1.0
        for item_id in reversed(context_items[-3:]):
            if 0 <= item_id < dense_transition.shape[0]:
                session_scores += weight * dense_transition[item_id]
            weight *= 0.8
        if session_scores.max() > 0:
            session_scores = session_scores / session_scores.max()

        blended = lambda_ * als_scores + (1 - lambda_) * session_scores
        seen = set(train_matrix[user_id].indices) | set(context_items)
        for seen_item in seen:
            if 0 <= seen_item < len(blended):
                blended[seen_item] = -np.inf
        recommendations.append(np.argsort(-blended)[:k])
        ground_truth.append(target_item)
    return np.array(recommendations), np.array(ground_truth)

for lambda_value in [0.0, 0.7, 1.0]:
    recs, targets = recommend_session_blend(als, transition, test_sessions, train_matrix, k=K, lambda_=lambda_value)
    print(f'lambda={lambda_value:.1f}: Recall@{K}={recall_at_k(recs, targets, K):.4f}')

> 💬 **Diskuter**
>
> 1. Hvorfor kan en kombinasjon av langtidsprofil og sesjon være bedre enn bare én av delene?
> 2. Hvem hjelper korttidskontekst mest?
> 3. Hva sier dette om hvorfor ekte systemer blir hybride?

## 🏋️ Oppgave 5 — Fairness og reranking

Høy gjennomsnittlig Recall betyr ikke at alle brukere er fornøyde.
Se på grafen under — hvem tjener, og hvem taper?

In [ ]:
bias_df = popularity_bias_analysis(recs_als, item_pop_counts, k=K)
group_df = group_recall_comparison(recs_als, test_items, train_matrix, item_pop_counts, K)

print(bias_df)

fig, ax = plt.subplots(figsize=(10, 5))
width = 0.35
x = range(len(bias_df))
ax.bar([idx - width / 2 for idx in x], bias_df['share_catalogue'], width, label='Andel i katalog')
ax.bar([idx + width / 2 for idx in x], bias_df['share_recommended'], width, label='Andel i anbefalinger')
ax.set_xticks(list(x))
ax.set_xticklabels(bias_df['bin'], rotation=20, ha='right')
ax.set_ylabel('Andel')
ax.set_title('Popularitetsbias: katalog vs anbefalinger')
ax.legend()
plt.tight_layout()
plt.show()

print('Recall@K per brukergruppe (ALS):')
print(group_df.to_string(index=False))

In [ ]:
def apply_mmr(als_model, user_ids, train_matrix, genre_matrix, k=10, lambda_=0.6, n_cand=50):
    candidate_ids, candidate_scores = als_model.recommend(user_ids, train_matrix[user_ids], N=n_cand, filter_already_liked_items=True)
    recommendations = np.zeros((len(user_ids), k), dtype=np.int32)
    for row_index in range(len(user_ids)):
        recommendations[row_index] = mmr_rerank(candidate_ids[row_index], candidate_scores[row_index], genre_matrix, k=k, lambda_=lambda_)
    return recommendations

recs_mmr = apply_mmr(als, user_ids, train_matrix, genre_matrix, k=K, lambda_=0.6)

rows = []
for name, recs in [('ALS', recs_als), ('ALS+MMR', recs_mmr)]:
    rows.append((name, recall_at_k(recs, test_items, K), coverage(recs, n_items, K), novelty(recs, item_pop_frac, K)))

print(f"{'Modell':<10} {'Recall@10':>10} {'Coverage':>10} {'Novelty':>10}")
print('-' * 46)
for name, recall_value, coverage_value, novelty_value in rows:
    print(f'{name:<10} {recall_value:>10.4f} {coverage_value:>10.4f} {novelty_value:>10.2f}')

Recall går litt ned. Coverage og novelty går opp. Det er en **eksplisitt tradeoff** — ikke en feil.

> 💬 **Diskuter**
>
> 1. Hvem taper på en ren ALS-modell?
> 2. Hva vinner vi og hva taper vi når vi re-rangerer for mangfold?
> 3. Hva ville du fortalt Amira om systemets styrker og svakheter akkurat nå?

---

> *Amira:* «Bra. Nå har dere vist at hybrider kan hjelpe, og at fairness må måles.
> Men det er én ting vi ikke har testet: hva skjer med alt dette når brukeren er helt ny?
> Og hva anbefaler dere faktisk at StreamNord shipper?»

**Neste steg** → `04_ship_decision.ipynb`

## Valgfri appendix — kalibrering

Hvis dere vil gå dypere i fairness, kan dere måle om anbefalingene matcher
brukerens faktiske sjangerprofil.

In [ ]:
cal_als, cal_mmr = [], []
for row_index, user_id in enumerate(user_ids[:500]):
    cal_als.append(genre_calibration_score(user_id, recs_als[row_index], train_matrix, genre_matrix))
    cal_mmr.append(genre_calibration_score(user_id, recs_mmr[row_index], train_matrix, genre_matrix))

print('Kalibrerings-KL (lavere = bedre):')
print(f'  ALS:     {np.mean(cal_als):.4f}')
print(f'  ALS+MMR: {np.mean(cal_mmr):.4f}')